
# Huggingfaceへのアップロード

## 事前準備

Hugging Faceからトークンを発行し、ログインしておく。
(初回のみ)

In [ ]:
!pip install -U --no-cache-dir pandas pyarrow tqdm imageio[ffmpeg] huggingface_hub 'numpy<2'

In [ ]:
HUGGINGFACE_TOKEN = ""

In [ ]:
%%capture
!hf auth login --token {HUGGINGFACE_TOKEN} --add-to-git-credential

## 設定、データモデル

In [ ]:
from enum import Enum

BASE_PATH = "./"
FPS = 30
HOME = "/home/jetson/notebooks/notebooks/"
NORMALIZE_FROM_0_224 = True
TASK_DESC = "Jetracer Driving task"


class CameraKey(str, Enum):
    FRONT_0 = "observation.images.front_0"
    FRONT_1 = "observation.images.front_1"

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import List


@dataclass
class CameraData:
    """単一カメラデータ"""

    task: str | None
    dataset: str | None

    @property
    def dataset_path(self):
        return Path(f"{BASE_PATH}/{self.task}/{self.dataset}")

    @property
    def input_root(self):
        return Path(HOME) / self.dataset_path

    @property
    def xy_path(self):
        return self.dataset_path / "xy"

    @property
    def speed_path(self):
        return self.dataset_path / "speed"

    @staticmethod
    def _file_count(p: Path) -> int:
        if p.is_dir():
            file_count = sum(f.is_file() for f in p.iterdir())
            return file_count
        else:
            return 0

    @property
    def xy_file_count(self):
        return self._file_count(self.xy_path)

    @property
    def speed_file_count(self):
        return self._file_count(self.speed_path)


@dataclass
class TwoCameraData:
    """2カメラデータ"""
    cam0: CameraData | None
    cam1: CameraData | None


@dataclass
class TwoCameraLeRobotDataset:
    """2カメラデータ用LeRobot Dataset"""
    episodes: List[TwoCameraData] = field(default_factory=list)


In [ ]:
out_dir = ""
output_dir_base = "lerobot_v3_out"
output_name: str | None = None
dataset = TwoCameraLeRobotDataset()
dataset.episodes.append(
    TwoCameraData(cam0=None, cam1=None)
)

## ビュー

In [ ]:
import ipywidgets
from IPython.display import clear_output
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label

l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

lineno = 0


def write_log(msg):
    global process_widget, lineno
    lineno = lineno + 1
    process_widget.value = str(lineno) + ": " + msg + "\n" + process_widget.value

    # UIのクリアと更新
    clear_output(wait=True)

In [ ]:
from pathlib import Path

import ipywidgets


class CameraDataView:
    model: CameraData = None

    def __init__(self):
        self.task_dropdown = ipywidgets.Dropdown(options=[], description='タスク')
        self.dataset_dropdown = ipywidgets.Dropdown(options=[], description='データセット')
        self.xy_data_count_widget = ipywidgets.IntText(description='XYデータ数')
        self.speed_data_count_widget = ipywidgets.IntText(description='速度データ数')

        self.dataset_dropdown.observe(self.update_count, names="value")
        self.task_dropdown.observe(self.change_dataset, names="value")
        self.change_task()

        self.container = ipywidgets.VBox([
            ipywidgets.HBox([self.dataset_dropdown, self.task_dropdown]),
            ipywidgets.HBox([self.xy_data_count_widget, self.speed_data_count_widget]),
        ])

    def set_model(self, model):
        self.model = model
        self.change_task()

    def update_count(self, change):
        if not self.model:
            return

        self.model.task = self.task_dropdown.value
        self.model.dataset = self.dataset_dropdown.value

        self.xy_data_count_widget.value = self.model.xy_file_count
        self.speed_data_count_widget.value = self.model.speed_file_count

        write_log(f"データセットを: {self.model.input_root}に設定")

    def change_dataset(self, change):
        if not self.model:
            return

        if not change["new"]:
            write_log("タスクが選択されていません。")
            return

        try:
            path = Path(BASE_PATH) / str(change["new"])
            write_log(f"change_dataset: {path}")
            if not path.exists():
                write_log(f"{path}が存在していません。")
                return

            dirs = [f.relative_to(path) for f in path.iterdir()
                    if f.is_dir()
                    and not str(f).startswith(".")
                    and str(f) not in {"__pycache__"}]
            dirs = sorted(dirs)
            self.dataset_dropdown.options = dirs
        except Exception as e:
            write_log(f"Error: {str(e)}")
            self.dataset_dropdown.options = []

    def change_task(self):
        if not self.model:
            return

        base_path = Path(BASE_PATH)
        if not base_path.exists():
            write_log(f"{base_path}が存在していません。")
            return

        try:
            dirs = [f.relative_to(base_path) for f in base_path.iterdir()
                    if f.is_dir()
                    and not str(f).startswith(".")
                    and str(f) not in {"__pycache__", "model_c", "model", "model_trt", "video", "zip"}]
            dirs = sorted(dirs)
            self.task_dropdown.options = dirs
        except Exception as e:
            write_log(f"Error: {str(e)}")
            self.task_dropdown.options = []


In [ ]:
cam0_view = CameraDataView()
cam1_view = CameraDataView()
output_name_input = ipywidgets.Text(description="データセット名")

## モデル=ビュー間のワイヤリング

In [ ]:
cam0_view.set_model(CameraData(task=None, dataset=None))
cam1_view.set_model(CameraData(task=None, dataset=None))
dataset.episodes[0].cam0 = cam0_view.model
dataset.episodes[0].cam1 = cam1_view.model


def output_name_input_on_changed(change):
    global output_name
    output_name = change["new"]


output_name_input.observe(output_name_input_on_changed, names="value")

## 必要なライブラリのImportとユーティリティの設定

In [ ]:
import json
import re
from dataclasses import dataclass
from typing import List, Tuple, Dict

import imageio.v3 as iio
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

try:
    import cv2  # optional

    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False


@dataclass
class FrameRec:
    idx: int
    path: Path
    steering_raw: float
    throttle_raw: float


def ensure_empty_dir(p: Path):
    # Remove and re-create output tree
    if p.exists():
        import shutil
        shutil.rmtree(p)
    (p / "meta").mkdir(parents=True, exist_ok=True)
    (p / "data" / "chunk-000").mkdir(parents=True, exist_ok=True)
    (p / "meta" / "episodes" / "chunk-000").mkdir(parents=True, exist_ok=True)
    for camera_key in [x.value for x in CameraKey.__members__.values()]:
        (p / "videos" / camera_key / "chunk-000").mkdir(parents=True, exist_ok=True)
    (p / "meta").mkdir(parents=True, exist_ok=True)


def parse_xy_index_from_name(name: str) -> Tuple[int | float, int | float, int] | None:
    # Extract x and y from filename: {x}_{y}_{index}.(png|jpg|jpeg)
    base = name.rsplit(".", 1)[0]
    m = re.search(r"^(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)_(\d+)$", base)
    if not m:
        return None
    x = float(m.group(1))
    y = float(m.group(2))
    index = int(m.group(3))
    if x.is_integer(): x = int(x)
    if y.is_integer(): y = int(y)
    return x, y, index


def find_xy_frames(input_root: Path) -> List[FrameRec]:
    xy_dir = input_root / "xy"
    if not xy_dir.is_dir():
        raise FileNotFoundError(f"xy/ not found: {xy_dir}")
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    files = (
        (p, *pr) for p in xy_dir.iterdir()
        if p.suffix.lower() in exts
           and (pr := parse_xy_index_from_name(p.name)) is not None
    )
    files = sorted(files, key=lambda f: f[3])
    frames: List[FrameRec] = []
    for f in files:
        p, x, y, index = f
        frames.append(FrameRec(idx=index, path=p, steering_raw=x, throttle_raw=y))
    return frames


def to_actions(frames: List[FrameRec]) -> np.ndarray:
    # Map x->steering, y->speed_forward
    if NORMALIZE_FROM_0_224:
        def norm(v):
            return (float(v) / 112.0) - 1.0

        steer = np.array([norm(fr.steering_raw) for fr in frames], dtype=np.float32)
        speed = np.array([norm(fr.throttle_raw) for fr in frames], dtype=np.float32)
    else:
        steer = np.array([float(fr.steering_raw) for fr in frames], dtype=np.float32)
        speed = np.array([float(fr.throttle_raw) for fr in frames], dtype=np.float32)
    return np.stack([steer, speed], axis=1)  # [N,2]


def load_image(path: Path) -> np.ndarray:
    # RGB ndarray(H,W,3)
    if _HAS_CV2:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img
    else:
        return iio.imread(path)


def write_video(frames: List[FrameRec], out_path: Path, fps: int):
    # Encode mp4 (libx264, yuv420p)
    import imageio_ffmpeg as ffmpeg
    first = load_image(frames[0].path)
    h, w = int(first.shape[0]), int(first.shape[1])

    cmd = [ffmpeg.get_ffmpeg_exe(), "-y", "-loglevel", "error",
           "-f", "rawvideo", "-vcodec", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{w}x{h}", "-r", str(fps),
           "-i", "-", "-an",
           "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
           str(out_path)]
    import subprocess
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)

    try:
        for fr in tqdm(frames, desc="encode video", unit="frame"):
            img = load_image(fr.path)
            assert img.shape[0] == h and img.shape[1] == w
            proc.stdin.write(img.tobytes(order="C"))
    finally:
        proc.stdin.close()
        proc.wait()


def write_parquet(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, path)


def split_episodes(n_frames: int, episode_seconds: int | None, fps: int) -> List[Tuple[int, int, int]]:
    if episode_seconds is None:
        return [(0, 0, n_frames)]
    L = int(episode_seconds * fps)
    if L <= 0:
        return [(0, 0, n_frames)]
    out = []
    s = 0
    ep = 0
    while s < n_frames:
        t = min(n_frames, s + L)
        out.append((ep, s, t))
        ep += 1
        s = t
    return out


def compute_stats(actions: np.ndarray) -> Dict:
    return {
        "action": {
            "mean": actions.mean(axis=0).tolist(),
            "std": actions.std(axis=0).tolist(),
            "min": actions.min(axis=0).tolist(),
            "max": actions.max(axis=0).tolist(),
        }
    }


## episodes/tasks/info.jsonの出力

In [ ]:
def write_tasks_parquet(out_root: Path, task_text: str):
    df = pd.DataFrame({"task_index": [0]}, index=[task_text])
    path = out_root / "meta" / "tasks.parquet"
    df.to_parquet(path)


def write_episodes_parquet(
        out_root: Path,
        episodes: List[Tuple[int, int, int]],
        fps: int,
        # camera_key: str,
        # total_frames: int,
        # video_duration_s: float,
):
    rows = []
    for (ep_idx, s, t) in episodes:
        from_frame = s
        to_frame = t
        length = to_frame - from_frame
        video_dict = {}
        for camera_key in [x.value for x in CameraKey.__members__.values()]:
            video_dict |= {
                f"videos/{camera_key}/chunk_index": 0,
                f"videos/{camera_key}/file_index": 0,
                f"videos/{camera_key}/from_timestamp": (from_frame / float(fps)),
                f"videos/{camera_key}/to_timestamp": (to_frame / float(fps)),
            }
        rows.append({
            "episode_index": ep_idx,
            "length": length,
            "data/chunk_index": 0,
            "data/file_index": 0,
            "dataset_from_index": from_frame,
            "dataset_to_index": to_frame,
            **video_dict,
            "from_frame_index": from_frame,
            "to_frame_index": to_frame,
        })
    ep_df = pd.DataFrame(rows)
    path = out_root / "meta" / "episodes" / "chunk-000" / "file-000.parquet"
    write_parquet(ep_df, path)


def write_info_json(
        out_root: Path,
        # camera_key: str,
        H: int, W: int,
        fps: int,
        total_frames: int,
        total_episodes: int,
        codec_str: str = "h264",
        data_mb: int = 100,
        video_mb: int = 500,
):
    camera_dict = {}
    for camera_key in [x.value for x in CameraKey.__members__.values()]:
        camera_dict |= {
            camera_key: {
                "dtype": "video",
                "shape": [H, W, 3],
                "names": ["height", "width", "channels"],
                "video_info": {
                    "video.height": H,
                    "video.width": W,
                    "video.channels": 3,
                    "video.codec": codec_str,
                    "video.pix_fmt": "yuv420p",
                    "video.is_depth_map": False,
                    "video.fps": float(fps),
                    "has_audio": False
                },
                "info": {
                    "video.height": H,
                    "video.width": W,
                    "video.channels": 3,
                    "video.codec": codec_str,
                    "video.pix_fmt": "yuv420p",
                    "video.is_depth_map": False,
                    "video.fps": int(fps),
                    "has_audio": False
                }
            }
        }
    info = {
        "codebase_version": "v3.0",
        "robot_type": "unknown",
        "total_episodes": total_episodes,
        "total_frames": total_frames,
        "total_tasks": 1,
        "chunks_size": 1000,
        "fps": int(fps),
        "splits": {"train": f"0:{total_episodes}"},
        "data_path": "data/chunk-{chunk_index:03d}/file-{file_index:03d}.parquet",
        "video_path": "videos/{video_key}/chunk-{chunk_index:03d}/file-{file_index:03d}.mp4",
        "features": {
            **camera_dict,
            "action": {
                "dtype": "float32",
                "shape": [2],
                "names": ["steering", "speed"],
                "fps": int(fps)
            },
            "timestamp": {
                "dtype": "float32",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "frame_index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "episode_index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "task_index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            }
        },
        "data_files_size_in_mb": int(data_mb),
        "video_files_size_in_mb": int(video_mb),
    }
    (out_root / "meta").mkdir(parents=True, exist_ok=True)
    with (out_root / "meta" / "info.json").open("w") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)


## 変換処理

In [ ]:
convert_button = ipywidgets.Button(description='LeRobot形式に変換')


def convert(change):
    global out_dir

    if not output_name:
        write_log(f"データセット名を指定してください")
        return

    out_dir = Path(BASE_PATH) / output_dir_base / output_name
    ensure_empty_dir(out_dir)

    # 1) Read frames & build actions
    # TODO: 複数エピソード対応
    camera_keys = [x.value for x in CameraKey.__members__.values()]
    video_tasks = zip(camera_keys, [dataset.episodes[0].cam0, dataset.episodes[0].cam1])
    total_frames = 0
    H = None
    W = None
    actions = None
    fps_meta = None
    fps_int = None
    for video_task in video_tasks:
        camera_key = video_task[0]
        camera_data = video_task[1]

        frames = find_xy_frames(camera_data.input_root)
        frame_length = len(frames)
        if frame_length == 0:
            raise RuntimeError(f"camera_key={camera_key},task={camera_data.task},dataset={camera_data.dataset}: "
                               f"No {{ID_x_y.*}} images found under xy/.'")

        write_log(f"camera_key={camera_key},task={camera_data.task},dataset={camera_data.dataset}: "
                  f"frame length: {frame_length}")

        # 2) Image size
        first = load_image(frames[0].path)
        H, W = int(first.shape[0]), int(first.shape[1])
        write_log(f"camera_key={camera_key},task={camera_data.task},dataset={camera_data.dataset}: "
                  f"Image size: {W}x{H}")

        # 3) Write video at target FPS
        video_dir = out_dir / "videos" / camera_key / "chunk-000"
        video_dir.mkdir(parents=True, exist_ok=True)
        video_path = video_dir / "file-000.mp4"
        write_video(frames, video_path, fps=int(FPS))

        if camera_key == CameraKey.FRONT_0:
            total_frames += frame_length

            actions = to_actions(frames)  # [N,2]

            # 4) Read actual FPS from encoded video
            video_meta = {}
            try:
                video_meta = iio.immeta(str(video_path))
            except Exception:
                video_meta = {}
            fps_meta = float(video_meta.get("fps", FPS))
            fps_int = int(round(fps_meta))
            write_log(
                f"[INFO] Encoded video fps(meta)={video_meta.get('fps')} -> using FPS_META={fps_meta} (int={fps_int})")

    # 5) Episodes
    episodes = [(0, 0, total_frames)]
    total_episodes = len(episodes)
    write_log(f"Episodes: {total_episodes}")

    # 6) Write data parquet (timestamp per-episode from 0s)
    rows = []
    global_idx = 0
    for ep_idx, s, t in episodes:
        for local_i, g in enumerate(range(s, t)):
            rows.append({
                "action": np.asarray(actions[g], dtype=np.float32),
                "timestamp": np.float32(local_i / float(fps_meta)),
                "frame_index": np.int64(local_i),
                "episode_index": np.int64(ep_idx),
                "index": np.int64(global_idx),
                "task_index": np.int64(0)
            })
            global_idx += 1
    df = pd.DataFrame(rows)
    data_path = out_dir / "data" / "chunk-000" / "file-000.parquet"
    write_parquet(df, data_path)

    # 7) Write episodes parquet (seconds via actual fps)
    write_episodes_parquet(
        out_root=out_dir, episodes=episodes,
        fps=int(fps_int),
        # camera_key=camera_key,
        # total_frames=frame_length,
        # video_duration_s=frame_length / fps_meta,
    )

    # 8) stats / tasks / info
    stats = compute_stats(actions)
    with (out_dir / "meta" / "stats.json").open("w") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    write_tasks_parquet(out_dir, TASK_DESC)
    write_info_json(
        out_root=out_dir,
        # camera_key=camera_key,
        H=H, W=W, fps=int(fps_int),
        total_frames=total_frames, total_episodes=total_episodes,
        codec_str="h264", data_mb=100, video_mb=500)

    write_log(f"\n[OK] {out_dir}フォルダにLeRobot Dataset v3フォーマットで保存")


convert_button.on_click(convert)

## Push to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

upload_button = ipywidgets.Button(description='アップロード')
repo_name_widget = ipywidgets.Text(description='Repo Name')


def upload(change):
    api = HfApi()
    repo_id = repo_name = repo_name_widget.value
    write_log(f"アップロード開始 repo_id: {repo_id}...(時間がかかります)")
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True, private=False)
    api.upload_folder(
        repo_id=repo_id,
        repo_type="dataset",
        folder_path=str(out_dir),
        allow_patterns=["meta/**", "data/**", "videos/**", "README.md"],
    )
    write_log(f"https://huggingface.co/datasets/{repo_id}")
    write_log(f"https://huggingface.co/spaces/lerobot/visualize_dataset?path={repo_id}")


upload_button.on_click(upload)

## Display Widgets

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')

title1 = ipywidgets.HTML('<b>【1.データセットを選択】</b> Huggingfaceにアップロードするデータセットを選択')
title1_sub1 = ipywidgets.HTML('<b>カメラ1</b>')
title1_sub2 = ipywidgets.HTML('<b>カメラ2</b>')
title2 = ipywidgets.HTML('<b>【2.LeRobot Datasetへ変換】</b> LeRobot Dataset v3へ変換')
title3 = ipywidgets.HTML('<b>【3.HuggingfaceにUpload】</b> Huggingfaceにアップロード')

data_collection_widget = ipywidgets.VBox([
    separator,
    title1,
    title1_sub1,
    cam0_view.container,
    title1_sub2,
    cam1_view.container,
    process_widget,
    title2,
    output_name_input,
    convert_button,
    process_widget,
    title3,
    ipywidgets.HBox([repo_name_widget, upload_button]),
    process_widget,
])
display(data_collection_widget)